# Deliverable 2 – Transformer Architectures for Depth Estimation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lukas-sek/Pose_Estimation/blob/main/depth_estimation_d2.ipynb)

Replaces the UNet baseline with three pretrained transformer encoders:
- **DeiT-Small** – plain ViT, non-hierarchical  
- **Swin-Tiny** – hierarchical shifted-window transformer  
- **EfficientFormer-L1** – hybrid Transformer–CNN  

Two decoder options: **simple bilinear upscaling** vs **lightweight FPN**.

Explorations:
- Patch size 8 vs 16 (DeiT)
- Transfer learning: full fine-tuning vs frozen backbone (linear probing)

## 1 – Setup

In [ ]:
import os
if not os.path.exists('/content/Pose_Estimation'):
    !git clone https://github.com/lukas-sek/Pose_Estimation.git /content/Pose_Estimation
%cd /content/Pose_Estimation

In [ ]:
!pip install -q timm opencv-python-headless

## 1b – Sample Dataset (no Drive needed)

Run this cell to auto-generate 10 synthetic samples inside the repo so the whole notebook runs without mounting Google Drive.  
Skip it when you have the real dataset on Drive and want to train properly.

In [ ]:
# ── Sample dataset generator ──────────────────────────────────────────────────
# Generates 10 synthetic 256×256 image/depth pairs so you can test the full
# pipeline without the real CLOTH3D++ data.
# Set USE_SAMPLE = False to use the real dataset from Google Drive instead.

USE_SAMPLE = True   # ← flip to False when using real data

SAMPLE_ROOT = '/content/Pose_Estimation/cloth3d/data_sample'
N_SAMPLES   = 10
SIZE        = 256

if USE_SAMPLE:
    import os, numpy as np, cv2

    os.makedirs(f'{SAMPLE_ROOT}/image', exist_ok=True)
    os.makedirs(f'{SAMPLE_ROOT}/depth', exist_ok=True)

    rng   = np.random.default_rng(42)
    names = [f'sample_{i}' for i in range(N_SAMPLES)]

    for i, name in enumerate(names):
        H = W = SIZE
        cx = W // 2 + rng.integers(-20, 20)
        cy = H // 2 + rng.integers(-20, 20)
        rx = int(rng.integers(W // 5, W // 3))
        ry = int(rng.integers(H // 3, H // 2))

        yy, xx = np.mgrid[0:H, 0:W]
        mask = ((xx - cx) / rx) ** 2 + ((yy - cy) / ry) ** 2 <= 1.0

        # Depth: smooth paraboloid inside ellipse, 0 background
        dist          = ((xx - cx) / rx) ** 2 + ((yy - cy) / ry) ** 2
        depth         = np.zeros((H, W), dtype=np.float32)
        depth[mask]   = (1.0 - dist[mask]) * 1.5 + 0.5   # ~[0.5, 2.0]

        # RGB: coloured ellipse + mild noise (simulates clothing texture)
        colour = rng.integers(60, 230, size=3).astype(np.float32)
        img    = np.zeros((H, W, 3), dtype=np.float32)
        for c in range(3):
            noise = rng.normal(0, 12, (H, W)).astype(np.float32)
            img[:, :, c][mask] = np.clip(colour[c] + noise[mask], 0, 255)

        cv2.imwrite(f'{SAMPLE_ROOT}/image/{name}.jpg',
                    cv2.cvtColor(img.astype(np.uint8), cv2.COLOR_RGB2BGR),
                    [cv2.IMWRITE_JPEG_QUALITY, 95])
        np.save(f'{SAMPLE_ROOT}/depth/{name}.npy', depth)

    # Split: 7 train / 2 val / 1 test
    splits = {'train.txt': names[:7], 'validation.txt': names[7:9], 'test.txt': names[9:]}
    for fname, split_names in splits.items():
        with open(f'{SAMPLE_ROOT}/{fname}', 'w') as f:
            f.write('\n'.join(split_names) + '\n')

    print(f'Sample dataset created at: {SAMPLE_ROOT}')
    print(f'  train=7  val=2  test=1  ({SIZE}×{SIZE} px)')

    # Override CONFIG so the rest of the notebook uses sample data
    # (CONFIG is defined in cell 2, but we patch it here for sample mode)
    _SAMPLE_CONFIG_OVERRIDES = {
        'root'       : SAMPLE_ROOT,
        'batch_size' : 2,
        'epochs'     : 3,
        'patience'   : 2,
        'pretrained' : False,   # skip ImageNet download for a quick smoke-test
    }
    print('\nSample-mode CONFIG overrides (will apply after Section 2 runs):')
    for k, v in _SAMPLE_CONFIG_OVERRIDES.items():
        print(f'  {k} = {v}')
else:
    _SAMPLE_CONFIG_OVERRIDES = {}
    print('USE_SAMPLE=False – will use real dataset from Google Drive.')

In [ ]:
# Skip this cell if USE_SAMPLE=True (no Drive needed)
if not USE_SAMPLE:
    from google.colab import drive
    drive.mount('/gdrive', force_remount=True)
else:
    print('Sample mode – skipping Drive mount.')

In [ ]:
# Unzip preprocessed data from Drive into the Colab VM.
# Only runs when USE_SAMPLE=False.

if not USE_SAMPLE:
    import zipfile, os

    ZIP_PATH    = '/gdrive/MyDrive/OR/Deliverable 3/preprocessed_data.zip'
    EXTRACT_TO  = '/content/Pose_Estimation/cloth3d/data'

    if os.path.isdir(EXTRACT_TO) and os.listdir(EXTRACT_TO):
        print(f'Data already extracted at {EXTRACT_TO} – skipping unzip.')
    else:
        print(f'Extracting {ZIP_PATH} → {EXTRACT_TO} …')
        os.makedirs(EXTRACT_TO, exist_ok=True)
        with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
            members = zf.namelist()
            # Strip a single top-level folder if the zip was created with one
            # e.g. preprocessed_data/image/... → image/...
            prefix = ''
            if members and all(m.startswith(members[0].split('/')[0] + '/') for m in members if '/' in m):
                prefix = members[0].split('/')[0] + '/'

            for member in members:
                stripped = member[len(prefix):]
                if not stripped:
                    continue
                dest = os.path.join(EXTRACT_TO, stripped)
                if member.endswith('/'):
                    os.makedirs(dest, exist_ok=True)
                else:
                    os.makedirs(os.path.dirname(dest), exist_ok=True)
                    with zf.open(member) as src, open(dest, 'wb') as dst:
                        dst.write(src.read())

        print('Done. Contents:')
        for entry in sorted(os.listdir(EXTRACT_TO)):
            n = len(os.listdir(f'{EXTRACT_TO}/{entry}')) if os.path.isdir(f'{EXTRACT_TO}/{entry}') else ''
            print(f'  {entry}/  {n}')
else:
    print('Sample mode – skipping unzip.')

In [ ]:
import os, gc, pickle, time
import numpy as np
import cv2
from matplotlib import pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import timm

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
print(f'timm version: {timm.__version__}')

## 2 – Configuration

Change `CONFIG` here to switch encoder, decoder, patch size, or transfer learning strategy before running the training cells.

In [ ]:
CONFIG = {
    # ── Data ──────────────────────────────────────────────
    'root'      : '/content/Pose_Estimation/cloth3d/data',   # real dataset path
    'img_size'  : 256,

    # ── Architecture ──────────────────────────────────────
    # encoder  : 'deit' | 'swin' | 'efficientformer'
    # decoder  : 'bilinear' | 'fpn'
    # patch_size: 8 | 16  (DeiT only; pretrained weights exist for 16)
    'encoder'    : 'deit',
    'decoder'    : 'bilinear',
    'patch_size' : 16,

    # ── Transfer learning ─────────────────────────────────
    # 'full_finetune'  – update all parameters
    # 'frozen_backbone' – freeze encoder; train decoder only
    'transfer'   : 'full_finetune',
    'pretrained' : True,

    # ── Training ──────────────────────────────────────────
    'batch_size'    : 8,
    'lr'            : 1e-4,
    'weight_decay'  : 1e-4,
    'epochs'        : 30,
    'patience'      : 5,

    # ── Output paths ──────────────────────────────────────
    'checkpoint'   : '/content/best_model_d2.pth',
    'history_path' : '/content/history_d2.pkl',
}

# Apply sample-mode overrides if USE_SAMPLE was set in section 1b
if '_SAMPLE_CONFIG_OVERRIDES' in dir() and _SAMPLE_CONFIG_OVERRIDES:
    CONFIG.update(_SAMPLE_CONFIG_OVERRIDES)
    print('Running in SAMPLE mode:', {k: CONFIG[k] for k in _SAMPLE_CONFIG_OVERRIDES})
else:
    print('Running with real dataset at:', CONFIG['root'])

## 3 – Data

In [ ]:
def read_list(path):
    with open(path) as f:
        return [l.strip() for l in f if l.strip()]

root = CONFIG['root']
train_list = read_list(f'{root}/train.txt')
val_list   = read_list(f'{root}/validation.txt')
test_list  = read_list(f'{root}/test.txt')

print(f'Train: {len(train_list)}  Val: {len(val_list)}  Test: {len(test_list)}')

In [ ]:
class Cloth3DDataset(Dataset):
    """PyTorch dataset for CLOTH3D++ depth estimation."""

    def __init__(self, data_list, root, img_size=256, augment=False):
        self.data_list = data_list
        self.root      = root
        self.img_size  = img_size
        self.augment   = augment

    def __len__(self):
        return len(self.data_list)

    def __getitem__(self, idx):
        name = self.data_list[idx]

        # ── Depth ─────────────────────────────────────────
        dpt  = np.load(f'{self.root}/depth/{name}.npy').astype(np.float32)
        mask = dpt > 0
        if mask.any():
            dpt[mask] = (dpt[mask] - dpt[mask].min() + 0.001) / 2.0

        # ── RGB ───────────────────────────────────────────
        img = cv2.imread(f'{self.root}/image/{name}.jpg')
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32)

        # ── Resize ────────────────────────────────────────
        s = self.img_size
        if img.shape[:2] != (s, s):
            img  = cv2.resize(img,  (s, s))
            dpt  = cv2.resize(dpt,  (s, s), interpolation=cv2.INTER_NEAREST)
            mask = dpt > 0

        # ── Per-image normalisation (foreground only) ─────
        if mask.any():
            for c in range(3):
                ch = img[:, :, c]
                mu, sigma = ch[mask].mean(), ch[mask].std()
                img[:, :, c] = (ch - mu) / (sigma + 1e-5)

        # ── Augmentation ──────────────────────────────────
        if self.augment:
            # Horizontal flip
            if np.random.rand() > 0.5:
                img = img[:, ::-1, :].copy()
                dpt = dpt[:, ::-1   ].copy()
            # Random translation (±20 px)
            tx, ty = np.random.randint(-20, 20, 2).tolist()
            M = np.float32([[1, 0, tx], [0, 1, ty]])
            img = cv2.warpAffine(img, M, (s, s))
            dpt = cv2.warpAffine(dpt, M, (s, s))

        img = torch.from_numpy(img).permute(2, 0, 1)  # (3, H, W)
        dpt = torch.from_numpy(dpt).unsqueeze(0)       # (1, H, W)
        return img, dpt

## 4 – Encoder Architectures

All encoders accept an RGB image `(B, 3, H, W)` and return a **list of feature maps**  
(single-scale for DeiT, multi-scale for Swin / EfficientFormer).

In [ ]:
class DeiTEncoder(nn.Module):
    """
    DeiT-Small encoder.
    Patch size 16 → pretrained ImageNet weights available.
    Patch size 8  → architecture overridden, random initialisation
                    (ImageNet pretrained weights are patch-16-specific).
    Output: [(B, 384, H/p, W/p)]  (single scale)
    """
    def __init__(self, patch_size=16, pretrained=True, img_size=256):
        super().__init__()
        use_pretrained = pretrained and (patch_size == 16)
        if pretrained and patch_size != 16:
            print(f'[DeiT] No pretrained weights for patch_size={patch_size} – using random init.')

        kwargs = dict(
            pretrained=use_pretrained,
            img_size=img_size,
            num_classes=0,
        )
        if patch_size != 16:
            kwargs['patch_size'] = patch_size

        self.backbone   = timm.create_model('deit_small_patch16_224', **kwargs)
        self.embed_dim  = self.backbone.embed_dim          # 384 for DeiT-Small
        self.out_channels = [self.embed_dim]

    def forward(self, x):
        B = x.shape[0]
        feats   = self.backbone.forward_features(x)        # (B, 1+N, D)
        patches = feats[:, 1:]                             # remove CLS  (B, N, D)
        N = patches.shape[1]
        H = W = int(N ** 0.5)
        spatial = patches.reshape(B, H, W, -1).permute(0, 3, 1, 2)  # (B, D, H, W)
        return [spatial]


class SwinEncoder(nn.Module):
    """
    Swin-Tiny hierarchical encoder.
    Output: 4 feature maps at strides 4, 8, 16, 32.
    Channels: [96, 192, 384, 768]
    """
    def __init__(self, pretrained=True, img_size=256):
        super().__init__()
        self.backbone = timm.create_model(
            'swin_tiny_patch4_window7_224',
            pretrained=pretrained,
            features_only=True,
            img_size=img_size,
        )
        self.out_channels = self.backbone.feature_info.channels()
        print(f'[Swin] Feature channels: {self.out_channels}')

    def forward(self, x):
        return self.backbone(x)   # list of 4 tensors


class EfficientFormerEncoder(nn.Module):
    """
    EfficientFormer-L1 hybrid Transformer-CNN encoder.
    Output: multi-scale feature list.
    """
    def __init__(self, pretrained=True, img_size=256):
        super().__init__()
        self.backbone = timm.create_model(
            'efficientformer_l1',
            pretrained=pretrained,
            features_only=True,
        )
        self.out_channels = self.backbone.feature_info.channels()
        print(f'[EfficientFormer] Feature channels: {self.out_channels}')

    def forward(self, x):
        return self.backbone(x)

## 5 – Decoder Architectures

Both decoders accept the list of feature maps from the encoder and output a single-channel depth map `(B, 1, H, W)` normalised to `[0, 1]`.

In [ ]:
class BilinearDecoder(nn.Module):
    """
    Simple decoder: bilinear-upsample the deepest feature map to target
    resolution, then refine with three conv layers.
    """
    def __init__(self, in_channels_list, target_size=256):
        super().__init__()
        self.target_size = target_size
        in_ch = in_channels_list[-1]

        self.refine = nn.Sequential(
            nn.Conv2d(in_ch, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.Conv2d(256,   128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.Conv2d(128,    64, 3, padding=1), nn.BatchNorm2d(64),  nn.ReLU(inplace=True),
            nn.Conv2d(64,      1, 1),
            nn.Sigmoid(),
        )

    def forward(self, features):
        x = features[-1]
        x = F.interpolate(x, size=(self.target_size, self.target_size),
                          mode='bilinear', align_corners=False)
        return self.refine(x)


class FPNDecoder(nn.Module):
    """
    Lightweight Feature Pyramid Network decoder.
    Projects every scale to fpn_channels with 1×1 convs, merges top-down,
    upsamples all levels to target size, concatenates and predicts.
    Works with single-scale input (DeiT) as a degenerate FPN.
    """
    def __init__(self, in_channels_list, target_size=256, fpn_channels=128):
        super().__init__()
        self.target_size = target_size

        self.laterals = nn.ModuleList([
            nn.Conv2d(c, fpn_channels, 1) for c in in_channels_list
        ])
        self.out_convs = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(fpn_channels, fpn_channels, 3, padding=1),
                nn.ReLU(inplace=True),
            )
            for _ in in_channels_list
        ])

        n = len(in_channels_list)
        self.head = nn.Sequential(
            nn.Conv2d(fpn_channels * n, 128, 3, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.Conv2d(128, 64, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(64,   1, 1),
            nn.Sigmoid(),
        )

    def forward(self, features):
        lats = [l(f) for l, f in zip(self.laterals, features)]

        # Top-down merge
        for i in range(len(lats) - 1, 0, -1):
            lats[i - 1] = lats[i - 1] + F.interpolate(
                lats[i], size=lats[i - 1].shape[-2:],
                mode='bilinear', align_corners=False,
            )

        # Upsample all scales to target and concatenate
        ups = [
            F.interpolate(conv(lat), size=(self.target_size, self.target_size),
                          mode='bilinear', align_corners=False)
            for conv, lat in zip(self.out_convs, lats)
        ]
        return self.head(torch.cat(ups, dim=1))

## 6 – Combined Model and Transfer Learning

In [ ]:
class DepthModel(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, x):
        return self.decoder(self.encoder(x))


def build_model(cfg):
    enc_name   = cfg['encoder']
    pretrained = cfg['pretrained']
    img_size   = cfg['img_size']

    if enc_name == 'deit':
        encoder = DeiTEncoder(patch_size=cfg['patch_size'],
                              pretrained=pretrained, img_size=img_size)
    elif enc_name == 'swin':
        encoder = SwinEncoder(pretrained=pretrained, img_size=img_size)
    elif enc_name == 'efficientformer':
        encoder = EfficientFormerEncoder(pretrained=pretrained, img_size=img_size)
    else:
        raise ValueError(f'Unknown encoder: {enc_name}')

    if cfg['decoder'] == 'bilinear':
        decoder = BilinearDecoder(encoder.out_channels, target_size=img_size)
    elif cfg['decoder'] == 'fpn':
        decoder = FPNDecoder(encoder.out_channels, target_size=img_size)
    else:
        raise ValueError(f'Unknown decoder: {cfg["decoder"]}')

    return DepthModel(encoder, decoder)


def apply_transfer_learning(model, mode):
    """
    full_finetune  : all weights trainable (standard fine-tuning).
    frozen_backbone: freeze encoder entirely; only decoder is updated
                     (linear probing / feature-extractor mode).
    """
    if mode == 'full_finetune':
        for p in model.parameters():
            p.requires_grad = True
    elif mode == 'frozen_backbone':
        for p in model.encoder.parameters():
            p.requires_grad = False
        for p in model.decoder.parameters():
            p.requires_grad = True
    else:
        raise ValueError(f'Unknown transfer mode: {mode}')

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(f'Transfer learning : {mode}')
    print(f'Trainable params  : {trainable:,} / {total:,}  ({100*trainable/total:.1f}%)')

## 7 – Metrics

- **RMSE** – root mean squared error on foreground pixels  
- **Mean angular error** – angle between predicted and GT surface normals  
  (normals computed as cross-product of depth gradients, i.e. finite differences)

In [ ]:
def rmse(pred, gt):
    """RMSE on foreground (depth > 0) pixels."""
    mask = gt > 0
    if not mask.any():
        return torch.tensor(0.0, device=pred.device)
    return torch.sqrt(F.mse_loss(pred[mask], gt[mask]))


def depth_to_normals(depth, eps=1e-6):
    """
    Estimate surface normals from a depth map via central finite differences.
    depth : (B, 1, H, W)
    returns (B, 3, H, W) unit normals
    """
    dz_dx = depth[:, :, :, 2:] - depth[:, :, :, :-2]   # (B,1,H,W-2)
    dz_dy = depth[:, :, 2:, :] - depth[:, :, :-2, :]   # (B,1,H-2,W)
    dz_dx = F.pad(dz_dx, [1, 1, 0, 0])
    dz_dy = F.pad(dz_dy, [0, 0, 1, 1])
    ones  = torch.ones_like(dz_dx)
    n = torch.cat([-dz_dx, -dz_dy, ones], dim=1)        # (B, 3, H, W)
    return n / (torch.norm(n, dim=1, keepdim=True) + eps)


def mean_angular_error(pred_depth, gt_depth):
    """Mean angular error (degrees) between surface normals, foreground only."""
    mask = (gt_depth > 0).squeeze(1)                    # (B, H, W)
    n_pred = depth_to_normals(pred_depth)
    n_gt   = depth_to_normals(gt_depth)
    cos_sim = (n_pred * n_gt).sum(dim=1).clamp(-1.0, 1.0)  # (B, H, W)
    angle   = torch.acos(cos_sim) * (180.0 / torch.pi)     # degrees
    if not mask.any():
        return torch.tensor(0.0, device=pred_depth.device)
    return angle[mask].mean()

## 8 – Training

In [ ]:
def train_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0.0
    for X, Y in loader:
        X, Y = X.to(device), Y.to(device)
        pred = model(X)
        loss = F.mse_loss(pred, Y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def val_epoch(model, loader, device):
    model.eval()
    total_rmse, total_mae_n, n = 0.0, 0.0, 0
    for X, Y in loader:
        X, Y = X.to(device), Y.to(device)
        pred = model(X)
        total_rmse  += rmse(pred, Y).item()
        total_mae_n += mean_angular_error(pred, Y).item()
        n += 1
    return total_rmse / n, total_mae_n / n


def train(model, cfg, train_list, val_list):
    train_ds = Cloth3DDataset(train_list, cfg['root'], cfg['img_size'], augment=True)
    val_ds   = Cloth3DDataset(val_list,   cfg['root'], cfg['img_size'], augment=False)
    train_dl = DataLoader(train_ds, batch_size=cfg['batch_size'], shuffle=True,
                          num_workers=2, pin_memory=True)
    val_dl   = DataLoader(val_ds,   batch_size=cfg['batch_size'], shuffle=False,
                          num_workers=2, pin_memory=True)

    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=cfg['lr'], weight_decay=cfg['weight_decay'],
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=cfg['epochs']
    )

    best_rmse, patience_counter = float('inf'), 0
    history = {'train_loss': [], 'val_rmse': [], 'val_mae_n': []}

    for epoch in range(cfg['epochs']):
        t0 = time.time()
        tr_loss         = train_epoch(model, train_dl, optimizer, DEVICE)
        v_rmse, v_mae_n = val_epoch(model, val_dl, DEVICE)
        scheduler.step()

        history['train_loss'].append(tr_loss)
        history['val_rmse'  ].append(v_rmse)
        history['val_mae_n' ].append(v_mae_n)

        print(f'Epoch {epoch+1:3d}/{cfg["epochs"]} | '
              f'loss={tr_loss:.4f} | val_RMSE={v_rmse:.4f} | '
              f'val_NormalMAE={v_mae_n:.2f}° | {time.time()-t0:.1f}s')

        if v_rmse < best_rmse:
            best_rmse = v_rmse
            torch.save(model.state_dict(), cfg['checkpoint'])
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= cfg['patience']:
                print(f'Early stopping at epoch {epoch+1} (best val RMSE {best_rmse:.4f})')
                break

    with open(cfg['history_path'], 'wb') as fh:
        pickle.dump(history, fh)

    print(f'\nBest val RMSE: {best_rmse:.4f}')
    return history

## 9 – Build, Configure and Train

In [ ]:
model = build_model(CONFIG).to(DEVICE)
apply_transfer_learning(model, CONFIG['transfer'])

total_params = sum(p.numel() for p in model.parameters())
print(f'Total parameters: {total_params:,}')

history = train(model, CONFIG, train_list, val_list)

## 10 – Evaluation

In [ ]:
@torch.no_grad()
def evaluate(model, test_list, cfg):
    model.eval()
    model.load_state_dict(torch.load(cfg['checkpoint'], map_location=DEVICE))

    test_ds = Cloth3DDataset(test_list, cfg['root'], cfg['img_size'], augment=False)
    test_dl = DataLoader(test_ds, batch_size=1, shuffle=False)

    total_rmse, total_mae_n = 0.0, 0.0
    fps_times = []

    if DEVICE == 'cuda':
        torch.cuda.reset_peak_memory_stats()

    for X, Y in test_dl:
        X, Y = X.to(DEVICE), Y.to(DEVICE)
        t0 = time.time()
        pred = model(X)
        if DEVICE == 'cuda':
            torch.cuda.synchronize()
        fps_times.append(time.time() - t0)
        total_rmse  += rmse(pred, Y).item()
        total_mae_n += mean_angular_error(pred, Y).item()

    n         = len(test_dl)
    avg_rmse  = total_rmse  / n
    avg_mae_n = total_mae_n / n
    fps       = 1.0 / (sum(fps_times) / len(fps_times))
    params    = sum(p.numel() for p in model.parameters())
    gpu_mb    = torch.cuda.max_memory_allocated() / 1024**2 if DEVICE == 'cuda' else 0

    print('=' * 52)
    print(f'  Test RMSE          : {avg_rmse:.4f}')
    print(f'  Normal MAE         : {avg_mae_n:.2f}°')
    print(f'  FPS                : {fps:.1f}')
    print(f'  Parameters         : {params:,}')
    print(f'  Peak GPU mem (MB)  : {gpu_mb:.1f}')
    print('=' * 52)

    return dict(rmse=avg_rmse, normal_mae=avg_mae_n,
                fps=fps, params=params, gpu_mb=gpu_mb)


results = evaluate(model, test_list, CONFIG)

## 11 – Visualisation

In [ ]:
hist = pickle.load(open(CONFIG['history_path'], 'rb'))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle(
    f"{CONFIG['encoder'].upper()} + {CONFIG['decoder']}  |  "
    f"patch={CONFIG['patch_size']}  |  TL={CONFIG['transfer']}",
    fontweight='bold'
)
axes[0].plot(hist['train_loss']); axes[0].set(title='Train Loss (MSE)',  xlabel='epoch')
axes[1].plot(hist['val_rmse'  ]); axes[1].set(title='Val RMSE',          xlabel='epoch')
axes[2].plot(hist['val_mae_n' ]); axes[2].set(title='Val Normal MAE (°)', xlabel='epoch')
plt.tight_layout()
plt.savefig('/content/training_curves_d2.png', dpi=150)
plt.show()

In [ ]:
@torch.no_grad()
def visualize_predictions(model, test_list, cfg, n_samples=4):
    model.eval()
    ds = Cloth3DDataset(test_list, cfg['root'], cfg['img_size'], augment=False)

    n_samples = min(n_samples, len(ds))   # clamp to available samples
    if n_samples == 0:
        print('No test samples to visualise.')
        return

    fig, axes = plt.subplots(n_samples, 4, figsize=(16, 4 * n_samples))
    if n_samples == 1:
        axes = axes[np.newaxis, :]   # keep 2-D indexing when only 1 row
    fig.suptitle('RGB  |  GT depth  |  Predicted depth  |  Normal error map',
                 fontweight='bold')

    for i in range(n_samples):
        X, Y = ds[i]
        pred = model(X.unsqueeze(0).to(DEVICE)).cpu().squeeze(0)   # (1,H,W)

        rgb = X.permute(1, 2, 0).numpy()
        rgb = (rgb - rgb.min()) / (rgb.max() - rgb.min() + 1e-5)

        gt_np   = Y.squeeze().numpy()
        pred_np = pred.squeeze().numpy()

        # Normal error map
        n_pred = depth_to_normals(pred.unsqueeze(0))
        n_gt   = depth_to_normals(Y.unsqueeze(0))
        cos_s  = (n_pred * n_gt).sum(dim=1).clamp(-1.0, 1.0)
        err_map = (torch.acos(cos_s) * 180.0 / torch.pi).squeeze().numpy()

        axes[i, 0].imshow(rgb)
        axes[i, 1].imshow(gt_np,   cmap='plasma')
        axes[i, 2].imshow(pred_np, cmap='plasma')
        im = axes[i, 3].imshow(err_map, cmap='hot', vmin=0, vmax=45)
        plt.colorbar(im, ax=axes[i, 3], fraction=0.046)
        for ax in axes[i]:
            ax.axis('off')

    plt.tight_layout()
    plt.savefig('/content/predictions_d2.png', dpi=150)
    plt.show()


visualize_predictions(model, test_list, CONFIG)

## 12 – Experiment Grid

Use `run_experiment` to systematically compare configurations.  
Fill in the RMSE / Normal MAE after each run.

| # | Encoder | Decoder | Patch | Transfer | RMSE ↓ | NormalMAE° ↓ | FPS ↑ | Params |
|---|---------|---------|-------|----------|--------|--------------|-------|--------|
| 1 | DeiT-S | Bilinear | 16 | Full fine-tune | | | | |
| 2 | DeiT-S | Bilinear |  8 | Full fine-tune | | | | |
| 3 | DeiT-S | FPN      | 16 | Full fine-tune | | | | |
| 4 | DeiT-S | Bilinear | 16 | Frozen backbone | | | | |
| 5 | Swin-T | FPN      | —  | Full fine-tune | | | | |
| 6 | EffFormer | FPN   | —  | Full fine-tune | | | | |

In [ ]:
def run_experiment(encoder, decoder, patch_size=16, transfer='full_finetune', pretrained=True):
    """Convenience wrapper: build → train → evaluate → return metrics dict."""
    tag = f'{encoder}_{decoder}_p{patch_size}_{transfer[:4]}'
    cfg = {
        **CONFIG,
        'encoder'      : encoder,
        'decoder'      : decoder,
        'patch_size'   : patch_size,
        'transfer'     : transfer,
        'pretrained'   : pretrained,
        'checkpoint'   : f'/content/model_{tag}.pth',
        'history_path' : f'/content/hist_{tag}.pkl',
    }
    m = build_model(cfg).to(DEVICE)
    apply_transfer_learning(m, transfer)
    train(m, cfg, train_list, val_list)
    return evaluate(m, test_list, cfg)


# ── Uncomment experiments one at a time or in sequence ────────────────────────

# Exp 1 – DeiT-S bilinear patch16 full fine-tune  (baseline for this deliverable)
# r1 = run_experiment('deit', 'bilinear', 16, 'full_finetune')

# Exp 2 – patch size comparison: patch 8 (no pretrained)
# r2 = run_experiment('deit', 'bilinear', 8, 'full_finetune')

# Exp 3 – decoder comparison: FPN
# r3 = run_experiment('deit', 'fpn', 16, 'full_finetune')

# Exp 4 – transfer learning: frozen backbone
# r4 = run_experiment('deit', 'bilinear', 16, 'frozen_backbone')

# Exp 5 – Swin-Tiny + FPN
# r5 = run_experiment('swin', 'fpn', 16, 'full_finetune')

# Exp 6 – EfficientFormer + FPN
# r6 = run_experiment('efficientformer', 'fpn', 16, 'full_finetune')

## 13 – Architecture Diagrams and Efficiency Summary

Prints parameter count, GMACs, and the layer-by-layer summary for each encoder.  
Use this to support the *memory vs speed vs performance* discussion in Q4.

In [ ]:
!pip install -q torchinfo
from torchinfo import summary

IMG = CONFIG['img_size']

arch_configs = [
    dict(encoder='deit',            decoder='bilinear', patch_size=16),
    dict(encoder='deit',            decoder='bilinear', patch_size=8),
    dict(encoder='swin',            decoder='fpn',      patch_size=16),
    dict(encoder='efficientformer', decoder='fpn',      patch_size=16),
]

arch_stats = []
for ac in arch_configs:
    cfg_tmp = {**CONFIG, **ac}
    m = build_model(cfg_tmp).to(DEVICE)
    m.eval()
    label = f"{ac['encoder'].upper()} p{ac['patch_size']} + {ac['decoder']}"
    print(f"\n{'='*60}\n  {label}\n{'='*60}")
    s = summary(m, input_size=(1, 3, IMG, IMG), verbose=1,
                col_names=['input_size', 'output_size', 'num_params', 'mult_adds'])
    arch_stats.append(dict(label=label, params=s.total_params, macs=s.total_mult_adds))
    del m; torch.cuda.empty_cache()

print('\n── Quick comparison ─────────────────────────────────────')
print(f'{"Architecture":<42} {"Params (M)":>12} {"GMACs":>8}')
print('-' * 65)
for a in arch_stats:
    print(f"{a['label']:<42} {a['params']/1e6:>11.2f}M {a['macs']/1e9:>7.2f}G")

## 14 – Transformer vs CNN Comparison

Fill `results_table` with your measured values after running all experiments, then run this cell to produce the comparison chart.

In [ ]:
import pandas as pd

# ── Fill in after running all experiments ─────────────────────────────────────
# (label, RMSE, NormalMAE°, FPS, Params_M, GPU_MB)
results_table = [
    ('UNet (CNN baseline)',           None, None, None, None, None),  # from D1
    ('DeiT-S Bilinear p16 full',      None, None, None, None, None),  # r1
    ('DeiT-S Bilinear p8  full',      None, None, None, None, None),  # r2
    ('DeiT-S FPN      p16 full',      None, None, None, None, None),  # r3
    ('DeiT-S Bilinear p16 frozen',    None, None, None, None, None),  # r4
    ('Swin-T FPN           full',     None, None, None, None, None),  # r5
    ('EffFormer FPN        full',     None, None, None, None, None),  # r6
]

cols = ['Model', 'RMSE ↓', 'NormalMAE° ↓', 'FPS ↑', 'Params (M)', 'GPU MB']
df = pd.DataFrame(results_table, columns=cols)
print(df.to_string(index=False))

# ── Bar charts ─────────────────────────────────────────────────────────────────
filled = [(r[0], r[1], r[2]) for r in results_table if r[1] is not None]
if filled:
    labels = [f[0] for f in filled]
    rmses  = [f[1] for f in filled]
    maes   = [f[2] for f in filled]
    colors = ['tab:blue' if 'UNet' in l else 'tab:orange' for l in labels]
    x = range(len(labels))

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    ax1.bar(x, rmses,  color=colors)
    ax1.set_xticks(x); ax1.set_xticklabels(labels, rotation=30, ha='right')
    ax1.set_ylabel('RMSE ↓'); ax1.set_title('Test RMSE: CNN vs Transformers')

    ax2.bar(x, maes, color=colors)
    ax2.set_xticks(x); ax2.set_xticklabels(labels, rotation=30, ha='right')
    ax2.set_ylabel('Normal MAE (°) ↓'); ax2.set_title('Normal MAE: CNN vs Transformers')

    plt.tight_layout()
    plt.savefig('/content/cnn_vs_transformer.png', dpi=150)
    plt.show()
else:
    print('No results yet – run experiments first and fill results_table.')

## 15 – Discussion Questions

Answer each question after completing all experiments. Reference specific experiment numbers and metrics where possible.

### Q1 – Do transformers preserve spatial detail?

*Hints: patch tokenisation discards sub-patch detail by design; global self-attention captures long-range context but lacks the local inductive bias of convolutions; compare patch-8 vs patch-16 RMSE and normal MAE — smaller patches recover finer boundaries; examine the normal error maps from Section 11 around cloth edges and body contours.*

**Your answer:**

> *(write here)*

### Q2 – Where do transformers fail vs CNNs?

*Hints: look at the qualitative prediction grid (Section 11) — are errors concentrated at garment edges, fine folds, or uniform flat regions? ViTs are data-hungry; with only ~200 training sequences the pretrained backbone matters more. Compare Exp 1 (pretrained) vs a scratch-trained run if you have one. Discuss background/foreground boundary artefacts caused by the zero-padded depth normalisation.*

**Your answer:**

> *(write here)*

### Q3 – What is most important for accuracy: transfer learning, patch size, or decoder design?

*Hints: isolate each factor using the controlled experiment pairs — Exp 1 vs Exp 4 (transfer learning only varies), Exp 1 vs Exp 2 (patch size only varies), Exp 1 vs Exp 3 (decoder only varies). Compute the RMSE delta for each pair. Which delta is largest? Why does ImageNet pretraining help especially on a small dataset? When might a frozen backbone be preferred despite lower accuracy?*

**Your answer:**

> *(write here)*

### Q4 – Architecture diagram, memory vs speed, and performance vs CNN

*Hints:*
- *Architecture pipeline: for each encoder describe the flow — patch embed → transformer blocks (self-attention type) → feature map reshape → decoder (lateral 1×1 → top-down FPN merge → bilinear upsample → conv head → sigmoid). Draw or describe the shapes at each stage.*
- *Memory vs speed: use the torchinfo table from Section 13. Which model has the best RMSE/GMAC trade-off? Which fits in a 15 GB Colab GPU at batch size 8?*
- *Performance vs CNN: use the comparison bar chart from Section 14. Does the best transformer beat UNet on RMSE? What is the FPS and parameter cost?*

**Your answer:**

> *(write here)*